# Data Cleaning

In this notebook we explain how we prepare our data before the visual graph creation

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
input_file = Path("simpsons_episodes.csv")
output_file = Path("simpsons_episodes_clean.csv")

In [ ]:
df = pd.read_csv(input_file)
df.head()

## Columns Selection

In [ ]:
important_cols = [
    "title",
    "season",
    "number_in_season",
    "number_in_series",
    "original_air_date",
    "imdb_rating",
    "imdb_votes",
    "us_viewers_in_millions",
]

df = df[important_cols].copy()
df.head()

## Type conversion

In [ ]:
print("Types before conversion:")
print(df.dtypes)

df["original_air_date"] = pd.to_datetime(df["original_air_date"], errors="coerce")

print("Types after conversion:")
print(df.dtypes)

## NA value check

In [ ]:
print("NA's per column:")
print(df.isna().sum())

print("Rows with at least one NA value:")
df[df.isna().any(axis=1)]

## Imputation and Filtering

In [ ]:
df = df[df["season"] != 28].copy()

df["us_viewers_in_millions"] = (
    df.groupby("season")["us_viewers_in_millions"]
    .transform(lambda s: s.interpolate())
)

df.head()

## Duplicate check

In [ ]:
duplicates = df[df.duplicated()]
print("Duplicates found:", len(duplicates))
duplicates

## Derivated Columns

In [ ]:
# 1. Separate the date into individual columns
df["air_year"] = df["original_air_date"].dt.year
df["air_month"] = df["original_air_date"].dt.month
df["weekday_num"] = df["original_air_date"].dt.weekday

#2. Join season and number_in_season
df["season_episode"] = (
    "S"
    + df["season"].astype(int).astype(str).str.zfill(2)
    + "E"
    + df["number_in_season"].astype(int).astype(str).str.zfill(2)
)

# 3. Number of episodes per season within the dataset
episodes_per_season = df.groupby("season")["number_in_season"].count()
df["episodes_in_season_dataset"] = df["season"].map(episodes_per_season)

df.head()

## Ordering

In [ ]:
df = df.sort_values(
    ["original_air_date", "season", "number_in_season"]
).reset_index(drop=True)

df

## Export clean CSV

In [ ]:
df.to_csv(output_file, index=False)
print(f"File saved in: {output_file}")